# Live trading — Turtle Mean Reversion 30 Min, Binance USD-M Perpetuals (piloto Coinbase50)

Cuaderno de orquestacion en vivo para el port de la estrategia de reversion a la media desde IBKR/CME
hacia Binance USDⓈ-M perpetuals, sobre el universo piloto de 5 monedas
(`qsCrypto/config/universe_coinbase50_binance.yaml`). Estructura identica a
`Turtle_MR_Intraday_Live.ipynb` (mismo bus de eventos, mismo `trade()`, mismo patron de Monitor/journal),
sustituyendo cada componente especifico de IBKR por su equivalente de Binance:

| Componente | IBKR | Binance |
|---|---|---|
| Datos | `IBKRPriceHandler` (bars[-2], contratos con rollover) | `BinanceKlinePriceHandler` (flag `is_closed` del propio kline, sin rollover — perpetuo) |
| Portfolio | `IBKRFuturesPortfolio` (multiplicador de contrato) | `BinancePerpetualsPortfolio` (unidades del activo base, sin multiplicador; funding) |
| Ejecucion | `IBKRExecutionHandler` (bracket nativo `transmit=False/True`) | `BinancePerpetualExecutionHandler` (STOP_MARKET+TAKE_PROFIT_MARKET tras el padre; puente de fills via user-data-stream) |
| Monitor | clase embebida en el notebook | `qsCrypto.monitor.Monitor` (modulo promovido) |

## 0. Veredicto del backtest — LEER ANTES DE TOCAR `RUN_LIVE`

`MeanReversion_Binance_Perp.ipynb` es la validacion independiente de esta configuracion sobre
~4.7 años de datos reales de Binance. Su veredicto explicito:

> **NO DESPLIEGUES CAPITAL** (5/10 criterios del checklist). El edge es negativo incluso con
> costes en cero (CAGR=-58.6% en BTCUSDT); la rejilla de walk-forward converge de forma ESTABLE
> (PBO=0) a un parametro "menos malo" que sigue sin ser rentable. El diagnostico es estructural
> (los parametros calibrados para CME MBT a 30 minutos no transfieren a la microestructura 24/7
> de un perpetuo cripto), no algo que mas optimizacion vaya a arreglar.

Este cuaderno queda armado como **ejercicio tecnico de integracion** (igual que
`Turtle_MR_Intraday_Live.ipynb` documenta para su propio caso de CME, que tampoco paso su propio
checklist) — sirve para validar que el cableado data->strategy->portfolio->execution->monitor
funciona de punta a punta contra el testnet de Binance, NO para desplegar capital real. `RUN_LIVE`
se deja en `False` por defecto; y aunque se active, debe apuntar SIEMPRE a
`BINANCE_TESTNET_API_KEY`/`BINANCE_TESTNET_SECRET_KEY`, nunca a las claves de produccion, hasta que
exista una validacion independiente con veredicto positivo.

## 1. Imports y parametros — declarados antes de abrir nada

In [ ]:
import json, logging, logging.config, os, queue, sys, threading, time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from qsCrypto.config.universe import load_universe
from qsCrypto.data.binance_klines import BinanceKlinePriceHandler
from qsCrypto.data.streaming import StreamingCryptoPrices
from qsCrypto.strategy.strategy import MeanReversionFadeStrategy
from qsCrypto.portfolio.binance_portfolio import BinancePerpetualsPortfolio
from qsCrypto.execution.execution import create_execution_handler
from qsCrypto.monitor import Monitor
from qsCrypto.trading.trading import trade
from qsCrypto import settings

logging.config.fileConfig(ROOT / "qsCrypto" / "logging.conf")
logger = logging.getLogger("qsCrypto.notebooks.binance_live")

In [ ]:
# ─────────── UNIVERSO — del YAML curado, no hardcodeado ───────────
PILOT_ONLY = True   # False para el universo curado completo (no recomendado sin re-validar)
UNI = load_universe(pilot_only=PILOT_ONLY)
SYMBOLS = UNI.symbols
print(f"Universo: {UNI.universe_name} -> {SYMBOLS}")

# ─────────── PARAMETROS CANONICOS (identicos al backtest, NO retocar aqui) ───────────
BAR_INTERVAL   = "30m"
ENTRY_BARS     = 20
ATR_PERIOD     = 20
STOP_N         = 1.0
ADD_N          = 0.5
MAX_UNITS      = 4
MAX_HOLD_BARS  = 10
TP_MODE        = "mid_channel"
RISK_PER_TRADE = Decimal("0.01")
MAX_GROSS_LEV  = 4.0
MAX_DIR_UNITS  = 12
MAX_CORR_UNITS = 6

# ─────────── SEGURIDAD PARA ESTA FASE (instruccion explicita del usuario) ───────────
# Ejercicio tecnico, no despliegue de capital: tope duro de notional por orden y
# sin piramidado (una posicion nunca arriesga mas de 1N).
MAX_ORDER_NOTIONAL_USDT = 200.0
MAX_UNITS_TESTNET = 1

TEST_MODE = True     # SIEMPRE True hasta pasar el go/no-go — testnet, nunca produccion
EQUITY    = Decimal(str(settings.EQUITY))
HEARTBEAT = 0.5

WARMUP_LIMIT = 500
JOURNAL_DIR = ROOT / "journals"
STATE_DIR = ROOT / "output"
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
JOURNAL = JOURNAL_DIR / f"mr_intraday_{UNI.universe_name}_{SESSION_ID}.jsonl"
STATE = STATE_DIR / f"live_state_mr_{UNI.universe_name}.json"

## 2. Cableado del motor

```
BinanceKlinePriceHandler   ->  BarEvent (barra cerrada, via flag is_closed del kline)
StreamingCryptoPrices      ->  TickEvent (bookTicker, para _ref_price/MAE)
MeanReversionFadeStrategy  ->  consume BAR, emite SignalEvent (sin cambios respecto a IBKR)
BinancePerpetualsPortfolio ->  consume SIGNAL, dimensiona (sin multiplicador), emite OrderEvent
                                con .stop_price Y .take_price (bracket nativo doble)
BinancePerpetualExecutionHandler -> consume ORDER, coloca MARKET + STOP_MARKET + TAKE_PROFIT_MARKET,
                                puente de FillEvent via user-data-stream
Monitor                    ->  recibe PortfolioEvent, escribe journal + snapshot atomico
```

In [ ]:
events = queue.Queue()

# ── datos: klines (barras) + bookTicker (ticks para ref-price/MAE) ──────────
prices = BinanceKlinePriceHandler(
    pairs=SYMBOLS, events_queue=events, interval=BAR_INTERVAL,
    test_mode=TEST_MODE, market_type=settings.BINANCE_MARKET_TYPE,
    warmup_limit=WARMUP_LIMIT,
)
prices.resolve_specs()
prices.warmup()

ticks = StreamingCryptoPrices(
    pairs=SYMBOLS, events_queue=events, test_mode=TEST_MODE,
    market_type=settings.BINANCE_MARKET_TYPE,
)

# ── estrategia — sembrada del historico de warmup, sin cambios vs. IBKR ─────
strategy = MeanReversionFadeStrategy(
    SYMBOLS, events, price_handler=prices, entry_bars=ENTRY_BARS,
    atr_period=ATR_PERIOD,
)

# ── ejecucion — bracket nativo + puente de fills ────────────────────────────
execution = create_execution_handler(
    test_mode=TEST_MODE, market_type=settings.BINANCE_MARKET_TYPE,
    broker="binance", events_queue=events,
)

# ── portfolio — sizing sin multiplicador, topes de seguridad de esta fase ───
portfolio = BinancePerpetualsPortfolio(
    prices, events, home_currency="USDT", equity=EQUITY,
    risk_per_trade=RISK_PER_TRADE, backtest=False, specs=prices.specs,
    stop_n=STOP_N, add_n=ADD_N, max_units=MAX_UNITS_TESTNET,
    max_hold_bars=MAX_HOLD_BARS, tp_mode=TP_MODE,
    max_gross_lev=MAX_GROSS_LEV, max_dir_units=MAX_DIR_UNITS,
    max_corr_units=MAX_CORR_UNITS, corr_groups=UNI.corr_groups_by_group(),
    execution=execution, strategy=strategy,
    max_order_notional=MAX_ORDER_NOTIONAL_USDT,
)

print(f"Especificaciones resueltas: {prices.specs}")
print(f"Cap de notional por orden: {MAX_ORDER_NOTIONAL_USDT} USDT | max_units={MAX_UNITS_TESTNET} (sin piramidado)")

## 3. Reconciliacion contra el broker (testnet)

Antes de arrancar los hilos: si el proceso se reinicio con una posicion aun abierta en la cuenta,
`reconcile_from_broker()` la reconstruye desde `positionRisk` + ordenes abiertas, en vez de operar
a ciegas sobre un estado local vacio.

In [ ]:
portfolio.reconcile_from_broker(execution)
print(f"Posiciones tras reconciliar: {list(portfolio.positions.keys())}")

## 4. Journal de la jornada

In [ ]:
monitor = Monitor(JOURNAL, STATE, session_id=SESSION_ID)
monitor.write(
    "session_start", symbols=SYMBOLS, equity=float(EQUITY),
    venue="binance_usdm", testnet=TEST_MODE, stop_n=STOP_N,
    entry_bars=ENTRY_BARS, risk=float(RISK_PER_TRADE),
    max_order_notional_usdt=MAX_ORDER_NOTIONAL_USDT,
    max_units=MAX_UNITS_TESTNET, corr_groups=UNI.corr_groups,
    funding_paid_to_date=float(portfolio.funding_paid),
    specs=prices.specs,
    backtest_verdict="NO DESPLIEGUES CAPITAL (ver MeanReversion_Binance_Perp.ipynb) — "
                     "esta sesion es un ejercicio tecnico de integracion, no despliegue de capital.",
)
print(f"Journal : {JOURNAL}")
print(f"Estado  : {STATE}")

## 5. Arrancar la sesion

⚠️ **Esta celda envia ordenes reales a la cuenta de TESTNET de Binance** (nunca produccion mientras
`TEST_MODE=True` arriba). `RUN_LIVE=False` dejalo montado sin enviar nada — es como conviene
arrancar la primera vez, y como debe quedar mientras el veredicto del backtest siga siendo
negativo.

In [ ]:
RUN_LIVE = False   # cambialo a True SOLO para una prueba deliberada en testnet

if not RUN_LIVE:
    print("RUN_LIVE=False — el motor no arranca. Nada se envia a Binance.")
else:
    if not TEST_MODE:
        raise RuntimeError(
            "TEST_MODE=False con el backtest en veredicto negativo: no se arranca contra produccion."
        )
    execution.start_user_data_stream()   # puente de fills, en su propio hilo via el thread de abajo

    trade_thread = threading.Thread(
        target=trade, args=(events, strategy, portfolio, execution, HEARTBEAT, monitor),
        daemon=True, name="trade-loop",
    )
    kline_thread = threading.Thread(target=prices.stream_to_queue, daemon=True, name="binance-klines")
    tick_thread = threading.Thread(target=ticks.stream_to_queue, daemon=True, name="binance-ticks")
    userdata_thread = threading.Thread(
        target=execution.start_user_data_stream, daemon=True, name="binance-userdata")

    for t in (trade_thread, kline_thread, tick_thread, userdata_thread):
        t.start()
    print("Hilos arrancados: trade-loop, binance-klines, binance-ticks, binance-userdata")

## 6. Monitor en vivo

Repinta cada `MONITOR_INTERVAL_S` segundos. Cruza el snapshot local del portfolio contra
`execution.get_position_info()` (fuente de verdad del broker) para detectar deriva.

In [ ]:
from IPython.display import display, clear_output

MONITOR_INTERVAL_S = 5

def render_once():
    clear_output(wait=True)
    print(f"Sesion {SESSION_ID} | hilos vivos: "
          f"trade={trade_thread.is_alive() if RUN_LIVE else '-'} "
          f"klines={kline_thread.is_alive() if RUN_LIVE else '-'} "
          f"ticks={tick_thread.is_alive() if RUN_LIVE else '-'}")
    print(f"Ultimo pulso: {monitor.last_heartbeat}")
    print(f"Posiciones locales: {portfolio.snapshot()}")
    if RUN_LIVE:
        try:
            broker_pos = [p for p in execution.get_position_info()
                          if abs(float(p.get('positionAmt', 0))) > 0]
            print(f"Posiciones en el broker (testnet): {broker_pos}")
        except Exception as exc:
            print(f"No se pudo leer el broker: {exc}")

# render_once()  # descomenta para una foto puntual
# while RUN_LIVE:
#     render_once()
#     time.sleep(MONITOR_INTERVAL_S)

## 7. Cierre de la jornada (kill switch)

Cancela lo vivo y aplana ANTES de cerrar la conexion — mismo orden que `IBKRExecutionHandler.flatten()`
y por la misma razon: un stop todavia vivo tras aplanar podria reabrir posicion contraria.

In [ ]:
def kill_all(reason="fin_de_sesion"):
    for s in SYMBOLS:
        try:
            flat = execution.flatten(s, reason=reason)
            monitor.write("kill", symbol=s, flat=flat, reason=reason)
            print(f"  {s}: aplanado {flat:+g}")
        except Exception as exc:
            print(f"  {s}: fallo al aplanar - {exc}")
    execution.stop_user_data_stream()
    prices.stop()
    ticks.stop()

# kill_all()   # descomenta para aplanar todo ahora mismo